In [ ]:
# =========================================================
# STEP 1: CONNECT GOOGLE DRIVE
# =========================================================

# Google Colab runs on a temporary computer.
# We connect Google Drive so our dataset and model are saved permanently.
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# =========================================================
# STEP 2: IMPORT REQUIRED LIBRARIES
# =========================================================

import pandas as pd
# Pandas helps us read and work with CSV files (tables of data)

import numpy as np
# NumPy helps us work with numbers and arrays (images are arrays of numbers)

import tensorflow as tf
# TensorFlow is the library that trains neural networks

from tensorflow.keras.utils import to_categorical
# This converts labels like 3 into [0 0 0 1 0 0 0]


In [ ]:
# =========================================================
# STEP 3: LOAD THE FER-2013 DATASET
# =========================================================

# The dataset contains:
# - emotion : number from 0 to 6
# - pixels  : face image stored as numbers in text form
data = pd.read_csv('/content/drive/MyDrive/fer2013.csv')


In [ ]:
# =========================================================
# STEP 4: SEPARATE IMAGES AND LABELS
# =========================================================

pixels = data['pixels'].tolist()
# pixels = INPUT data (face images written as numbers)

labels = data['emotion'].values
# labels = OUTPUT data (correct emotion for each image)


In [ ]:
# =========================================================
# STEP 5: CONVERT PIXEL TEXT INTO REAL IMAGES
# =========================================================

faces = []  # This will store actual face images

for pixel_string in pixels:
    # Each image is stored as one long string of numbers
    numbers = pixel_string.split()

    # Convert text numbers into real numeric values
    face = np.array(numbers, dtype='float32')

    # Convert 1D list into a 48×48 image
    # CNN understands images only in this form
    face = face.reshape(48, 48)

    faces.append(face)


In [ ]:
# =========================================================
# STEP 6: NORMALIZE AND PREPARE IMAGE DATA
# =========================================================

X = np.array(faces)
# Convert Python list into NumPy array (faster for math)

X = X / 255.0
# Pixel values go from 0–255
# We scale them to 0–1 to make learning easier

X = X.reshape(-1, 48, 48, 1)
# Shape explanation:
# - many images
# - height = 48
# - width = 48
# - 1 channel (grayscale)


In [ ]:
# =========================================================
# STEP 7: CONVERT EMOTION LABELS INTO MACHINE FORMAT
# =========================================================

y = to_categorical(labels, num_classes=7)
# Converts emotion number into one-hot format
# Example:
# Happy (3) → [0 0 0 1 0 0 0]


In [ ]:
# =========================================================
# STEP 8: SPLIT DATA INTO TRAIN AND TEST
# =========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Training data → model learns from this
# Test data → checks if model learned correctly


In [ ]:
# =========================================================
# STEP 9: BUILD THE CNN MODEL
# =========================================================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten


In [ ]:
model = Sequential([

    # This layer looks for edges and basic patterns in the face
    Conv2D(32, (3,3), activation='relu', input_shape=(48,48,1)),

    # Makes image smaller but keeps important information
    MaxPooling2D(2,2),

    # Randomly disables neurons to avoid memorizing data
    Dropout(0.25),

    # Learns more complex facial features
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Learns emotion-specific patterns
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Dropout(0.25),

    # Converts image data into a simple list of numbers
    Flatten(),

    # Combines all learned features to make a decision
    Dense(512, activation='relu'),
    Dropout(0.5),

    # Final layer gives probability of each emotion
    Dense(7, activation='softmax')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# =========================================================
# STEP 10: COMPILE THE MODEL
# =========================================================

model.compile(
    optimizer='adam',
    # Adam automatically adjusts learning speed

    loss='categorical_crossentropy',
    # Measures how wrong the prediction is

    metrics=['accuracy']
)


In [ ]:
# =========================================================
# STEP 11: TRAIN THE MODEL
# =========================================================

model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_test, y_test)
)

# The model:
# 1. Makes a guess
# 2. Checks how wrong it is
# 3. Fixes itself
# 4. Repeats many times


Epoch 1/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.2526 - loss: 1.8054 - val_accuracy: 0.3601 - val_loss: 1.6047
Epoch 2/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.3886 - loss: 1.5739 - val_accuracy: 0.4529 - val_loss: 1.4198
Epoch 3/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.4481 - loss: 1.4348 - val_accuracy: 0.4976 - val_loss: 1.3220
Epoch 4/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.4782 - loss: 1.3553 - val_accuracy: 0.5263 - val_loss: 1.2615
Epoch 5/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5043 - loss: 1.2988 - val_accuracy: 0.5369 - val_loss: 1.2223
Epoch 6/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.5222 - loss: 1.2500 - val_accuracy: 0.5500 - val_loss: 1.2033
Epoch 7/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5377 - loss: 1.2169 - val_accuracy: 0.5587 - val_loss: 1.1826
Epoch 8/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.5418 - loss: 1.1916 - val_accuracy: 

In [ ]:
# =========================================================
# STEP 12: SAVE THE TRAINED MODEL
# =========================================================

model.save('/content/drive/MyDrive/emotion_model.h5')

# This file contains everything the model learned
# We can now use it for real-time emotion detection
